# QSC: Production-Grade Post-Quantum Secure Channel
### Qiskit Fall Fest 2026 (QFF 2026) — Track 5: Post-Quantum Cryptography
**Problem Statement P1: Quantum-Safe Secure Channel**
**Codename:** `QSC — Quantum-Safe Channel`

---

## 1. Executive Summary & Architecture
This interactive notebook demonstrates the end-to-end execution, security verification, and performance analysis of **QSC**.
* **NIST FIPS 203 (ML-KEM-768):** Ephemeral Key Encapsulation Mechanism (PK: 1,184 B, CT: 1,088 B, SS: 32 B)
* **NIST FIPS 204 (ML-DSA-65):** Mutual Digital Signature Authentication (PK: 1,952 B, Sig: 3,309 B)
* **RFC 5869 HKDF-SHA256:** Context-separated session key derivation
* **NIST SP 800-38D AES-256-GCM:** 64 KB authenticated chunks with strictly monotonic 64-bit frame counter replay defense.
* **Zero Hand-Rolled Cryptography:** 100% compliant with standard reference libraries (liboqs-python and cryptography).

In [ ]:
import sys, os, asyncio
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in path
sys.path.insert(0, '.')

from src.problem import QuantumSafeChannelProblem, PQCKeyEncapsulation, PQCSignature
from src.processors import load_processor, list_available_processors
from src.routing import PacketRouter, MitMRoutingProxy
from src.metrics import compute_statistics
from src.visualization import generate_all_figures

print('QSC System Modules successfully loaded!')

## 2. NIST FIPS 203 & 204 Primitive Wire-Size Assertions
Assert exact primitive wire sizes at runtime without hand-rolled primitives.

In [ ]:
kem = PQCKeyEncapsulation('ML-KEM-768')
pk_kem, sk_kem = kem.generate_keypair()
ct_kem, ss_kem = kem.encap_secret(pk_kem)
ss_decap = kem.decap_secret(ct_kem, sk_kem)

assert ss_kem == ss_decap, 'KEM shared secret mismatch!'
print(f'[+] ML-KEM-768 Public Key:  {len(pk_kem):,} bytes (FIPS 203 exact: 1,184 B)')
print(f'[+] ML-KEM-768 Ciphertext:  {len(ct_kem):,} bytes (FIPS 203 exact: 1,088 B)')
print(f'[+] ML-KEM-768 Shared Secret: {len(ss_kem)} bytes (FIPS 203 exact: 32 B)')

sig = PQCSignature('ML-DSA-65')
pk_dsa, sk_dsa = sig.generate_keypair()
sample_msg = b'FIPS 204 Verification Vector 2026'
signature = sig.sign(sample_msg, sk_dsa)
valid = sig.verify(sample_msg, signature, pk_dsa)

assert valid is True, 'Signature verification failed!'
print(f'[+] ML-DSA-65 Public Key:   {len(pk_dsa):,} bytes (FIPS 204 exact: 1,952 B)')
print(f'[+] ML-DSA-65 Signature:    {len(signature):,} bytes (FIPS 204 exact: 3,309 B)')

## 3. End-to-End Secure Handshake & Streaming Session
Execute 1-RTT mutually authenticated handshake with transcript binding and stream encrypted payloads.

In [ ]:
problem = QuantumSafeChannelProblem()
test_payload = b'QFF_2026_TRACK5_PQC_PRODUCTION_GRADE_SECURE_PAYLOAD_' * 100

session_result = asyncio.run(problem.execute_secure_session(test_payload))
print('[*] Session Status:', session_result['status'])
print('[*] Payload Transferred:', session_result['payload_bytes'], 'bytes')
print('[*] Plaintext SHA-256:', session_result['sha256'])
print('[*] Handshake Latency:', f"{session_result['client_handshake']['latency_ms']:.3f} ms")
print('[*] Cryptographic Backend:', session_result['backend'])

## 4. Hardware Profiles (Processor A vs. Processor B)
Inspect reproducible processor constraint profiles in `processors/`.

In [ ]:
proc_a = load_processor('processor_A')
proc_b = load_processor('processor_B')

print(f"[Processor A] Name: {proc_a.name} | Cores: {proc_a.get_core_limit()} | RAM: {proc_a.memory_allocation.get('limit_human')}")
print(f"[Processor B] Name: {proc_b.name} | Cores: {proc_b.get_core_limit()} | RAM: {proc_b.memory_allocation.get('limit_human')}")

## 5. Headline Attack Demonstration (MitM Interception Proof)
* **Scenario A (Unauthenticated KEM):** Active proxy intercepts and decrypts all traffic.
* **Scenario B (QSC Authenticated):** Ephemeral key substitution fails transcript signature check and terminates safely.

In [ ]:
proxy = MitMRoutingProxy()
secret_doc = b'CONFIDENTIAL_PATENT_DOCUMENT_QFF2026_TRACK5'

res_a = asyncio.run(proxy.run_scenario_a_unauthenticated_kem(secret_doc))
print('--- Scenario A (Unauthenticated KEM) ---')
print('MitM Plaintext Intercepted:', res_a['intercepted_plaintext'])
print('Verdict:', res_a['security_verdict'])

nodes = problem.setup_identities('alice', 'bob')
from src.peer import QSCClient, QSCServer
client = QSCClient('alice', nodes['client']['sk'], nodes['client']['pk'], problem.client_trust_store)
server = QSCServer('bob', nodes['server']['sk'], nodes['server']['pk'], problem.server_trust_store)

res_b = asyncio.run(proxy.run_scenario_b_qsc_authenticated(problem.client_trust_store, problem.server_trust_store, client, server, secret_doc))
print('\n--- Scenario B (QSC Authenticated Channel) ---')
print('Handshake Aborted:', res_b['handshake_aborted'])
print('Plaintext Compromised:', res_b['plaintext_compromised'])
print('Verdict:', res_b['security_verdict'])

## 6. Empirical Systems Benchmark Results & Publication Figures
Visualizing generated charts from `results/figures/`.

In [ ]:
fig_paths = [
    'results/figures/fig1_handshake_wire_framing.png',
    'results/figures/fig2_handshake_latency.png',
    'results/figures/fig4_wire_overhead_pqc_vs_classical.png',
    'results/figures/fig5_throughput_vs_payload.png',
    'results/figures/fig6_platform_d_tradeoff.png',
]

for p in fig_paths:
    if os.path.exists(p):
        img = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(os.path.basename(p), fontsize=12, fontweight='bold')
        plt.show()
    else:
        print(f'Figure not found: {p}')